<a href="https://colab.research.google.com/github/kwanda2426/projects/blob/main/ML_In_Chem/Putting_it_together_Reg_1077167.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Boiling and Melting points Prediction** - Kwanda Mazibuko stdnr : 1077167

<div style="text-align: center;">
   <img src="https://raw.githubusercontent.com/kwanda2426/projects/main/ML_In_Chem/ML_and_Rdkit.png" >
</div>




# **Importing Libraries**

In [29]:
!pip -q install plotly scikit-learn tabulate pandas openpyxl rdkit pubchempy pandas

In [30]:
# Libraries for data loading, data manipulation and data visulisation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rc
import seaborn as sns
import plotly.express as px

from rdkit.Chem import Descriptors
from rdkit.Chem import Descriptors, rdMolDescriptors
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import rdchem
from rdkit.Chem import Draw
from rdkit.Chem import Draw

#Feature engineering, selection and Model training
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


#ignoring warnings
import warnings
warnings.filterwarnings('ignore')

#making sure that we can see all rows and cols
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
from tabulate import tabulate
from IPython.display import display


# **Data acquisition**

#### Front CSV File

In [31]:
# Reading data
url = "https://raw.githubusercontent.com/kwanda2426/projects/main/ML_In_Chem/hydrocarbons.csv"
df_1 = pd.read_csv(url)
df_1.head()

,Class of hydrocarbon,IUPAC name,Melting point,Boiling point,Density,Flash point,Autoignition temp,pubchem_id,smiles
0,Trimetylalkane,"2,2,4-Trimethylpentane",-107.0,99.0,0.69,NaN,396,10907,CC(C)CC(C)(C)C
1,Triaromatics,Phenanthrene,99.0,338.0,1.18,171,>450,995,C1=CC=C2C(=C1)C=CC3=CC=CC=C32
2,Triaromatics,Anthracene,216.0,341.0,1.2825,NaN,NaN,8418,C1=CC=C2C=C3C=CC=CC3=CC2=C1
3,Triaromatics,1-methylanthracene,86.0,342.0,1.04799,NaN,NaN,11884,CC1=CC=CC2=CC3=CC=CC=C3C=C12
4,Triaromatics,2-methylanthracene,209.0,340.0,1.8,NaN,NaN,11936,CC1=CC2=CC3=CC=CC=C3C=C2C=C1


#### Using RDKit - Molecular weight and Aromatic Rings

In [32]:
# Create new columns
df = df_1.copy()
df['Molecular_Weight'] = None
df['Aromatic_Rings'] = None

for i, smi in enumerate(df['smiles']):
    mol = Chem.MolFromSmiles(smi)
    if mol:
        # Calculate molecular properties using RDKit
        mol_weight = Descriptors.MolWt(mol)
        aromatic_rings = rdMolDescriptors.CalcNumAromaticRings(mol)

        # Store results back into the DataFrame
        df.loc[i, 'Molecular_Weight'] = mol_weight
        df.loc[i, 'Aromatic_Rings'] = aromatic_rings

df['Molecular_Weight'] = pd.to_numeric(df['Molecular_Weight'], errors='coerce')
df['Aromatic_Rings'] = pd.to_numeric(df['Aromatic_Rings'], errors='coerce')

df.head()

,Class of hydrocarbon,IUPAC name,Melting point,Boiling point,Density,Flash point,Autoignition temp,pubchem_id,smiles,Molecular_Weight,Aromatic_Rings
0,Trimetylalkane,"2,2,4-Trimethylpentane",-107.0,99.0,0.69,NaN,396,10907,CC(C)CC(C)(C)C,114.232,0
1,Triaromatics,Phenanthrene,99.0,338.0,1.18,171,>450,995,C1=CC=C2C(=C1)C=CC3=CC=CC=C32,178.234,3
2,Triaromatics,Anthracene,216.0,341.0,1.2825,NaN,NaN,8418,C1=CC=C2C=C3C=CC=CC3=CC2=C1,178.234,3
3,Triaromatics,1-methylanthracene,86.0,342.0,1.04799,NaN,NaN,11884,CC1=CC=CC2=CC3=CC=CC=C3C=C12,192.261,3
4,Triaromatics,2-methylanthracene,209.0,340.0,1.8,NaN,NaN,11936,CC1=CC2=CC3=CC=CC=C3C=C2C=C1,192.261,3


In [33]:
df.columns

Index(['Class of hydrocarbon', 'IUPAC name', 'Melting point', 'Boiling point',
       'Density', 'Flash point', 'Autoignition temp', 'pubchem_id', 'smiles',
       'Molecular_Weight', 'Aromatic_Rings'],
      dtype='object')

- Model is built on Molecular wight and aromatic Rings. Hence we keep only those columns.

#### Final Dataset

In [34]:
# columns to keep
cols = ['Molecular_Weight', 'Aromatic_Rings','Melting point', 'Boiling point']
df_final = df[cols]

# **Data Preprocessing**

Data preprocessing is vital because it transforms messy raw data into a clean, consistent, and structured format, which is essential for accurate analysis and effective machine learning models.
Issues like missing values, inconsistencies, errors, and outliers are addressed to improve data quality,leads to more reliable and meaningful insights for decision-making and ultimately enhances the performance and accuracy of models.



#### Reviewing and Renaming Columns


In [35]:
# data columns
df_final.columns

Index(['Molecular_Weight', 'Aromatic_Rings', 'Melting point', 'Boiling point'], dtype='object')

- There are column names that have 'spaces'/'dashes' as separators. These 'spaces'/'dashes' are  replaced with an 'underscore' for readability and data for simplicity in data manipulation. Also, all variable names are converted to lowercase.

In [36]:
# Changing column names
df_final.columns = df_final.columns.str.replace(' ', '_', regex=False).str.lower()
df_final.head()

,molecular_weight,aromatic_rings,melting_point,boiling_point
0,114.232,0,-107.0,99.0
1,178.234,3,99.0,338.0
2,178.234,3,216.0,341.0
3,192.261,3,86.0,342.0
4,192.261,3,209.0,340.0


#### Dataset Dimensions and Variable Type

In [37]:
df = df_final.copy()

In [38]:
# Rows and Columns of data
print('Data has {} rows and {} Columns'.format(df.shape[0],df.shape[1]))
print('')
print('Data has the following variables:')
print('')
for i in df.columns:
  missing_values = 100*df[i].isnull().sum()/df.shape[0]
  print(i+', with {}% of missing data'.format(missing_values))
  print('')

Data has 194 rows and 4 Columns

Data has the following variables:

molecular_weight, with 0.0% of missing data

aromatic_rings, with 0.0% of missing data

melting_point, with 3.0927835051546393% of missing data

boiling_point, with 2.0618556701030926% of missing data



In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   molecular_weight  194 non-null    float64
 1   aromatic_rings    194 non-null    int64  
 2   melting_point     188 non-null    float64
 3   boiling_point     190 non-null    float64
dtypes: float64(3), int64(1)
memory usage: 6.2 KB


- Dataset has some missing values which are less than 5%. These can be dropped.

In [40]:
df = df.dropna()

#### Summary Statistics and Variable Description

In [41]:
#describing numerical columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
molecular_weight,186.0,175.559500,113.639165,16.043,106.168,154.297,211.85325,843.636
aromatic_rings,186.0,0.478495,0.919500,0.000,0.000,0.000,1.00000,4.000
melting_point,186.0,-30.994624,82.013018,-188.000,-95.000,-27.500,15.00000,256.000
boiling_point,186.0,211.118280,138.702080,-162.000,115.500,206.500,300.25000,625.000


In [ ]:
# number of unique smiles and unique compound ids
print('There are {} unique smiles'.format(len(list(set(df.smiles.values)))))
print('There are {} unique coumpound ids'.format(len(list(set(df.compound_id.values)))))

- This shows that these variables are unique identifies for each data point.

In [ ]:
# Create the data as a list of dictionaries
variables = [

    {"variable": "compound_id", "description": "Unique identifier or name of the compound."},
    {"variable": "minimum_degree", "description": "Minimum number of connections (bonds) an atom in the molecule has."},
    {"variable": "molecular_weight", "description": "Molecular mass in atomic mass units (amu)."},
    {"variable": "number_of_h_bond_donors", "description": "Number of atoms in the molecule that can donate hydrogen bonds."},
    {"variable": "number_of_rings", "description": "Count of ring structures in the molecule."},
    {"variable": "number_of_rotatable_bonds", "description": "Flexibility of the molecule — high values indicate more rotatable bonds."},
    {"variable": "polar_surface_area", "description": "Surface area of polar atoms; affects solubility and permeability."},
    {"variable": "smiles", "description": "SMILES string — text representation of molecular structure."},
    {"variable": "measured_log_solubility_in_mols_per_litre", "description": "logS value indicating solubility - Target Variable."}

]


# Display the DataFrame
var_desc = pd.DataFrame(variables)
var_desc_final = var_desc.reset_index(drop = True)

# Display the DataFrame
print(tabulate(var_desc_final, headers = 'keys', tablefmt = 'grid'))


- Compound Id and smiles can be dropped because they are unique to the data points.

# **Feature Engineering**

This is a vital step where data is explored and then transformed from raw data into features that enhance machine learning model performance, accuracy, and generalization by making patterns more discernable.

### **Exploratory Data Analysis**

This method is meant to uncover the underlying structure of a data set and is important for a company because it exposes trends, patterns, and relationships that are not readily apparent.

#### **Feature Distribution**
This is meant to discover :
- Relationship between features and target variable. This should help see if a feature that was not caught by correlation because of datatype(especially categorical) has a relationship with the target variable. This helps in encoding categorical features. For an example, with the region in our dataset. It could be that the feature does not affect charges at all, or some regions have more impact than others.

In [ ]:
# dropping unnecessary columns
df_1 = df.copy()
df_1 = df_1.drop(['compound_id','smiles'], axis = 1)

#### Target Variable Distribution - log solubility

In [ ]:
sns.histplot(df_1['measured_log_solubility_in_mols_per_litre'], kde=True)
plt.title('Distribution of Log Solubility')
plt.show()

#### **Obsevations**

- Skewness: A right-skewness suggest that most compounds have low solubility, with a few having very high solubility. The skewness could reflect real-world instances where most compounds are poorly soluble, but a few are highly soluble due to specific molecular properties.

- Potential outliers: The long tail suggests there are some compounds with unusually high solubility. These must be handled carefully.

#### Correlation

In [ ]:
df_temp = df_1.copy()
plt.figure(figsize=(10,8))
sns.heatmap(df_temp.corr(), annot=True,cmap='coolwarm' )
plt.title('Feature Correlation Heatmap')
plt.show()

**Observations**
1. Highly Positively Correlated Features (*Multicollinearity Risk)

*This happens where the features in are linearly dependent. This means that they might carry the same information.
- polar_surface_area <-> molecular_weight (0.48)
- polar_surface_area <-> number_of_h_bond_donors (0.76)
- molecular_weight <-> number_of_rings (0.65)


This implies that these features might be tightly linked, likely carrying similar information.
2. Negatively Correlated Features with the Target Variable

- molecular_weight (-0.64): Heavier molecules tend to be less soluble. This is a strong signal—definitely a key predictor.
- number_of_rings (-0.51): More rings = lower solubility. Likely due to rigidity and reduced interaction with solvents.
- number_of_rotatable_bonds (-0.24): More flexibility might reduce solubility, but this is a weaker effect.

These negative correlations are useful in highlighting feature contrasts for better decision boundaries for our model.

3. Positive correlation with the Target Variable
- number_of_h_bond_donors (0.21): More hydrogen bond donors slightly increase solubility—makes sense chemically.
- polar_surface_area (0.12).


**Considerations**

- One of the highly correlated features can be dropped, Keeping all may cause multicollinearity. We may consider dropping polar_surface_area because it has weak correlation with the target variable.
-  Keep molecular_weight and number_of_rings: These are the top predictors of solubility.










#### Variables Distribution

In [ ]:
# columns for the pair plot
numerical_cols = df_1.columns
# Create the pair plot
sns.pairplot(df_1[numerical_cols], diag_kind = 'kde', corner = True)
plt.suptitle('Pair Plot of Numerical Features', y = 1.02)
plt.show()



- Most of the features look to be discrete. This means that they exist in certain ranges. Feature distributions seem to be showing feature skewness.

- Molecular Weight and Polar Surface Area: right-skewed, suggesting most compounds are small or moderately sized, with a few outliers that are much larger.
- Number of H-Bond Donors, Number of Rings, and Number of Rotatable Bonds: These may show discrete distributions (e.g., peaks at integers), reflecting structural constraints in molecular design.
- Measured Log Solubility: If this is roughly normal or slightly skewed, it gives insight into the solubility profile of your compound set—important for drug-likeness

#### skewness/kurtosis

In [ ]:
df_temp = df_1.copy()
features = df_temp.drop('measured_log_solubility_in_mols_per_litre', axis = 1)

# Calculate kurtosis for each feature
kurtosis_values = {
    'Feature': [],
    'Kurtosis': [],
    'Interpretation': [],
    'Kurtosis Value Interpretation': []
}

k_meaning = '<3: Light tails (fewer outliers), ~3: Normal distribution, >3: Heavy tails (more outliers)'

for col in features.columns:
    k = kurtosis(df_temp[col], fisher=True, bias=False)
    kurtosis_values['Feature'].append(col)
    kurtosis_values['Kurtosis'].append(round(k, 3))
    kurtosis_values['Kurtosis Value Interpretation'].append(k_meaning)
    if k > 3:
        interpretation = "Heavy tails (many outliers)"
    elif k < 3:
        interpretation = "Light tails (few outliers)"
    else:
        interpretation = "Normal-like tails"
    kurtosis_values['Interpretation'].append(interpretation)

# Display the results
kurtosis_df = pd.DataFrame(kurtosis_values)
print(tabulate(kurtosis_df, headers='keys', tablefmt='grid'))



4 features exhibit heavy-tailed distributions (kurtosis > 3), indicating high outlier risk, so a scaler robust to outliers is needed. The RobustScaler would be suitable for feature scaling.

**Key Takeaways**

- The target variable is already scaled (log), hence no scaling required.
- A RobustScaler is appropriate for most features.

From the Exploratory Data Analysis, polar_surface_area can be dropped.




In [ ]:
# final dataframe
df_final = df_1.drop('polar_surface_area', axis = 1)
df_final.head()

#### Feature Scaling

a standard scaler is used because it scales all features to have a mean of zero and a standard deviation of one, ensuring each feature contributes equally to the model.

it prevents features with larger numerical values from disproportionately influencing the model's learning.
When features have vastly different ranges (e.g., one feature ranges from 0-10 and another from 0-10,000), the model can become biased toward the feature with the larger magnitude, simply because its values are larger.

In [ ]:
# features and target variable
X = df_final.drop(['measured_log_solubility_in_mols_per_litre'], axis = 1)
y = df_final['measured_log_solubility_in_mols_per_litre']

# Initialize the scaler
scaler = RobustScaler()
# apply scaler to predictor variables
X_scaled = scaler.fit_transform(X)

#### Data Split
Splitting data into training and testing sets is essential to evaluate a model's performance on new, unseen data and prevent overfitting. The training set teaches the model patterns, while the untouched test set provides an unbiased assessment of its generalisation ability, ensuring the model performs accurately in real-world scenarios rather than just on the data it learned from.


In [ ]:
 # First plit: 80% train and 20% test -- random state = 42, ensure the reproducibility of the data split
X_temp, X_test, y_temp, y_test = train_test_split(X_scaled, y, test_size = 0.2, random_state = 42)

# Second split: train and validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.25, random_state = 42)

# **Model selection**

1. **Support Vector Regression (SVR)**
 :  is a strong choice for this solubility prediction task, especially with 1,128 data points. SVR handles nonlinear relationships well using kernel functions like RBF, which is ideal since molecular features mostly interact in complex ways. It is also robust to overfitting in smaller datasets because of its optimization capability, and it performs well when features vary in scale. With these different features and moderate dataset size, SVR offers a balance of flexibility and generalization that is suitable for this task.

 2. **Random Forest Regressor** : is a strong candidate for your solubility prediction task with 1,128 rows, as it excels at capturing nonlinear relationships and complex feature interactions without requiring extensive preprocessing. It is robust to outliers and multicollinearity, which is valuable given the variability of these features in this dataset. In addition, Random Forest provides built-in feature importance scores, helping to identify which features mostly influence solubility. Its ensemble nature also reduces overfitting, making it well-suited for medium-sized datasets.

 3. **Linear Regression** : is a good model for this solubility prediction task with 1,128 rows. It is simple, fast to implement. It is easy to interpret, this means that it ideal for benchmarking and understanding how each molecular feature contributes to solubility. If the relationships between features and solubility are approximately linear, it can perform well. Even when the fit is not perfect, Linear Regression helps reveal multicollinearity and feature significance, guiding feature selection and engineering for more complex models.

# **Model Training**

In this step, data is training data fed to the model. Training a model is essential because it allows the algorithm to learn patterns and relationships from data so it can make accurate predictions or decisions on new, unseen inputs. Without training, the model would have no understanding of how features relate to outcomes.

#### Support Vector Regression

In [ ]:
# initialise a SVR
svr = SVR(kernel='rbf', C = 1.0, epsilon = 0.1)

# train model
svr.fit(X_train, y_train)

#### Random Forest Regressor

In [ ]:
# Initialise a Random Forest Regressor
rf = RandomForestRegressor(n_estimators = 100, random_state = 42)

# Train model
rf.fit(X_train, y_train)


#### Linear Regression

In [ ]:
# Initialise a Linear Regression model
lr = LinearRegression()

# Train model
lr.fit(X_train, y_train)



# **Model Evaluation**

Models are evaluated using the validation dataset. Validating these trained models is crucial because it helps identify which one performs best to unseen data—without touching the test set. This step acts like a rehearsal before the final performance: it reveals whether the model is overfitting (memorising the training data) or underfitting (failing to learn enough). By evaluating on the validation set, we can select the most promising model and fine-tune its hyperparameters, ensuring that when it is finally tested, the true predictive power is measured — not just luck or overtraining.

### **Regression Evaluation Metrics**

 **Mean Squared Error (MSE)** - it measures the average of the squared differences between predicted and actual values. it is useful because it penalises larger errors more heavily, which is helpful when we want to reduces prediction errors.Since solubility can vary widely, MSE helps ensure that the  model does not perform badly on extreme values.

**Root Mean Squared Error (RMSE)** - it measures the square root of MSE. It puts error back in the same units as the target variable. it is useful because it is easier to interpret than MSE because it says that, on average, how far off the predictions are in terms of solubility. In this
case RMSE gives a direct sense of how accurate the solubility predictions are.

**Coefficient of Determination (R-Squared)** - it measures the proportion of variance in the target variable explained by the model. This is useful because it tells us how well the features explain solubility. A higher R-suared would mean our molecular features are capturing the factors that influence solubility well.

In essence, MSE and RMSE quantify prediction accuracy and R-squared explains model effectiveness in capturing underlying patterns.

In [ ]:
# Calculate kurtosis for each feature
metrics_ = {
    'Model': [],
    'MSE': [],
    'RMSE': [],
    'R-Squared': []
}

# SVR Metrics
y_pred = svr.predict(X_val)
metrics_['Model'].append('SVR')
metrics_['MSE'].append(mean_squared_error(y_val, y_pred))
metrics_['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_pred)))
metrics_['R-Squared'].append(r2_score(y_val, y_pred))

# Random Forest Regressor Metrics
y_pred = rf.predict(X_val)
metrics_['Model'].append('Random Forest')
metrics_['MSE'].append(mean_squared_error(y_val, y_pred))
metrics_['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_pred)))
metrics_['R-Squared'].append(r2_score(y_val, y_pred))

# Linear Regression Metrics
y_pred = lr.predict(X_val)
metrics_['Model'].append('Linear Regression')
metrics_['MSE'].append(mean_squared_error(y_val, y_pred))
metrics_['RMSE'].append(np.sqrt(mean_squared_error(y_val, y_pred)))
metrics_['R-Squared'].append(r2_score(y_val, y_pred))

# Display the results
metrics_df = pd.DataFrame(metrics_)
print(tabulate(metrics_df, headers = 'keys', tablefmt = 'grid'))

 **Interpretation of Metrics**

- MSE and RMSE: Lower values mean better prediction accuracy. ***Random Forest*** has the lowest value for both.
- R-squared Score: Measures how well the model explains variance in solubility. ***Random Forest*** shows the strongest fit.

This implies that Random Forest is capturing complex interactions between features most effectively.

#### Random Forest's Feature Importance

- These are the features that contributes towards to the model's predictive ability.

In [ ]:
# Extract feature importances from trained Random Forest model
importances = rf.feature_importances_
feature_names = X.columns

# Create a DataFrame
features_df = pd.DataFrame({
    'Feature Name': feature_names,
    'Importance Score': importances
})

# Sort by importance
features_df = features_df.sort_values(by='Importance Score', ascending=False)
features_df.reset_index(drop = True, inplace = True)

# Display the results
print(tabulate(features_df, headers='keys', tablefmt='grid'))

The Random Forest model identified several molecular descriptors that significantly influenced its ability to predict solubility. The most dominant feature was molecular_weight (importance score: 0.6879), indicating that the size of the molecule plays a crucial role in determining solubility—likely due to its impact on diffusion and interaction with solvents. Number_of_h_bond_donors (0.1087) followed as a key contributor, reflecting the molecule’s potential for hydrogen bonding, which directly affects aqueous solubility. Number_of_rotatable_bonds (0.0986) also emerged as important, suggesting that molecular flexibility may facilitate better solvent interactions. Number_of_rings (0.0861) added predictive value, possibly due to its influence on molecular rigidity and hydrophobicity. Lastly, minimum_degree (0.0187) had minimal impact, implying that atomic connectivity alone offers limited insight into solubility behavior within this dataset.

# **Hyperparameter Tuning of Random Forest Model**

Hyperparameter tuning is a critical step in refining a machine learning model after initial validation. While default settings can yield decent performance, they are rarely optimal for a specific dataset. By systematically adjusting parameters such as the number of trees, tree depth, and minimum samples required to split or form a leaf , we can significantly improve the model’s accuracy, precision, and generalisation. This process helps prevent overfitting or underfitting by tailoring the model to the unique patterns and complexity of the data. In essence, hyperparameter tuning ensures that the model is at its best.

In [ ]:
%%time
# initialise base model
rf = RandomForestRegressor(random_state = 42)

# hyperparameter grid
param_grid = {
    'n_estimators': [100, 200, 300],           # Number of trees
    'max_depth': [None, 10, 20, 30],           # Maximum depth of each tree
    'min_samples_split': [2, 5, 10],           # Minimum samples to split a node
    'min_samples_leaf': [1, 2, 4],             # Minimum samples at a leaf node
    'max_features': ['auto', 'sqrt', 'log2']   # Number of features to consider at each split
}

# Set up GridSearchCV
grid_search = GridSearchCV(estimator = rf,
                           param_grid = param_grid,
                           cv = 5,
                           scoring = 'explained_variance',#'neg_mean_squared_error',
                           n_jobs = -1,
                           verbose = 2)

# Training on combined data
grid_search.fit(X_temp, y_temp)

# Best model and parameters
best_rf = grid_search.best_estimator_
print("Best Parameters:", grid_search.best_params_)


#### Initialising and Training Tuned Model

In [ ]:
# initialise the best model
final_rf = RandomForestRegressor(max_depth =  10, max_features = 'sqrt', min_samples_leaf = 1, min_samples_split = 5, n_estimators = 100)

# train model on combined training data
final_rf.fit(X_temp,y_temp)

#### Evaluating Tuned Model


In [ ]:
# Calculate kurtosis for each feature
metrics_ = {
    'Model': [],
    'MSE': [],
    'RMSE': [],
    'R-Squared': []
}

# Random Forest Regressor Metrics
y_pred = final_rf.predict(X_test)
metrics_['Model'].append('Random Forest')
metrics_['MSE'].append(mean_squared_error(y_test, y_pred))
metrics_['RMSE'].append(np.sqrt(mean_squared_error(y_test, y_pred)))
metrics_['R-Squared'].append(r2_score(y_test, y_pred))

# Display the results
metrics_df = pd.DataFrame(metrics_)
print(tabulate(metrics_df, headers = 'keys', tablefmt = 'grid'))

The comparison between the tuned and untuned Random Forest models reveals subtle but meaningful differences in predictive performance. The untuned model achieved an MSE of 1.16933, RMSE of 1.08136, and an R-squared score of 0.73374, indicating strong baseline accuracy and variance explanation. After hyperparameter tuning, the MSE slightly increased to 1.257 and RMSE to 1.12116, while the R-squared score improved to 0.73407. This suggests that while the tuned model may produce slightly larger average errors, it generalises much better across data and captures more of the underlying variance in solubility. Given its consistent performance and ability to explain over 73% of the variation in solubility, the tuned Random Forest model is a reliable and suitable choice for predicting solubility.

# **Challenges and Potential Improvements**

 **Challenges Encountered**
- Feature Scaling Sensitivity : SVR required careful feature scaling, unlike Random Forest and Linear Regression. A robust scaler was used, but other options like a Box-Cox have the potential to improve performance. Unsuitable type of a scaler could lead to poor performance or misleading results.

- Nonlinear Relationships
This dataset has nonlinear relationships among the features, and most models struggle to capture complex patterns between molecular features and solubility.
This was evident on lower R-squared scores and higher error metrics.
- Hyperparameter Tuning : SVR and Random Forest both benefit significantly from tuning. But this is dependent on what hyperparameters are chosen to compare against.
- Interpretability vs. Performance
Random Forest performed best but is less interpretable than Linear Regression.
and explaining predictions to non-technical stakeholders may be harder.

**Potential Improvements**
 - Feature Engineering : features like: (Flexibility Index =  Number_of_rotatable_bond/molecular_weight) and (Hydrogen Bonding Potential = number_of_h_bond_donors × polar_surface_area) can be created. These may capture hidden patterns and boost model performance.
- Model Ensemble : Combine predictions from multiple models (like averaging SVR and Random Forest outputs). Ensembles often outperform individual models.

- Feature Selection : Use techniques like Recursive Feature Elimination (RFE).
- Nested Cross-Validation :
Use nested cross-validation to tune hyperparameters and evaluate model performance simultaneously. This avoids data leakage between tuning and testing phases and gives a more unbiased estimate of generalization error.
- Bootstrapping : Apply bootstrapping to generate multiple resampled datasets and assess the variability of the model’s predictions.





# **Conclusion**

Based on the results, we can conclude that the tuned Random Forest model is performing reliably and effectively on the solubility prediction task—achieving strong regression metrics and offering a clear, interpretable ranking of feature importance. With an R-squared score of 0.73407, the model explains over 73% of the variance in solubility, while maintaining a reasonable RMSE of 1.12116 and MSE of 1.257. These results suggest the model has successfully captured the underlying relationships between molecular descriptors and solubility.

What the model shows:
- Strong predictive performance on a chemically meaningful dataset.
- Clear interpretability through feature importance scores.
- Alignment with domain knowledge, reinforcing model credibility.

**What can be done to further test the model**:

- External Validation with a New Dataset :
Tuned model can be applied to a completely different solubility dataset or  from a different chemical library.

- Model Calibration :
Residual plots or calibration curves can be used to assess whether the regression predictions are systematically biased, such as  consistency in overestimating low-solubility compounds.
- Uncertainty Quantification :
Techniques like Quantile Regression Forests or Prediction Intervals can be used to estimate confidence bounds around each prediction.
- Domain-Specific Feature Enrichment :
Incorporate cheminformatics features like LogP, TPSA, or Lipinski’s Rule of Five indicators. These are known to correlate with solubility and may improve interpretability and performance.


# **References**

- James, G., Witten, D., Hastie, T. and Tibshirani, R., 2013. An Introduction to Statistical Learning: with Applications in R. New York: Springer.
- Géron, A., 2019. Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow: Concepts, Tools, and Techniques to Build Intelligent Systems. 2nd ed. Sebastopol: O'Reilly Media.
- Raschka, S. and Mirjalili, V., 2019. Python Machine Learning: Machine Learning and Deep Learning with Python, scikit-learn, and TensorFlow 2. 2nd ed. Birmingham: Packt Publishing.
- Towards Data Science, 2020. Understanding Feature Importance in Random Forests. [online] Available at: https://towardsdatascience.com/understanding-feature-importance-in-random-forests-264f187cf6f9
- Scikit-learn Developers, 2023. GridSearchCV Documentation. [online] Available at: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html
- Copilot.

- Meinshausen, N., 2006. Quantile Regression Forests. Journal of Machine Learning Research, 7, pp.983–999. Available at: https://jmlr.org/papers/volume7/meinshausen06a/meinshausen06a.pdf

- SCFBio-IIT Delhi, n.d. Lipinski Rule of Five. [online] Available at: https://scfbio-iitd.res.in/software/drugdesign/lipinski.jsp